# Modular Hamiltonian Demo
This notebook demonstrates the pluggable Hamiltonian and cross-pollination features:
- modular operator registration (relativistic correction)
- stochastic thermal noise
- operator-form non-unitary / Lindblad collapse (QuTiP if available)
- symbolic Noether-style commutator check (SymPy)

Run cells sequentially.

In [ ]:
import numpy as np
from physics_engine import EngineConfig, PhysicsEngine

# Build engine (start with use_qutip=False to exercise NumPy path)
cfg = EngineConfig(N=20, use_qutip=False)
eng = PhysicsEngine(cfg)

# Add a small relativistic correction (operator form)
eng.add_relativistic_mass_term(eps=1e-4)

# Add thermal noise and a Lindblad-like lowering operator
eng.add_thermal_noise(temperature=0.1, friction=0.01)
A, Ad = __import__('physics_engine').operators.ladder_ops(cfg.N)
eng.add_non_unitary_operator(A, strength=0.01)

psi0 = np.zeros(cfg.N, dtype=complex)
psi0[2] = 1.0
times = np.linspace(0.0, 0.2, 41)
res = eng.simulate(psi0, times)
print('Ran NumPy path: states', res.states.shape, 'energies', res.energies.shape)

In [ ]:
# Try the QuTiP backend if available (set use_qutip=True)
cfg_q = EngineConfig(N=12, use_qutip=True)
eng_q = PhysicsEngine(cfg_q)
A, Ad = __import__('physics_engine').operators.ladder_ops(cfg_q.N)
eng_q.add_non_unitary_operator(A, strength=0.02)
psi0_q = np.zeros(cfg_q.N, dtype=complex); psi0_q[3] = 1
times_q = np.linspace(0.0, 0.05, 11)
res_q = eng_q.simulate(psi0_q, times_q)
print('QuTiP/NumPy path result shapes:', res_q.states.shape, res_q.energies.shape)

In [ ]:
# Analytic harmonic oscillator benchmark
from physics_engine.hamiltonian import build_H
from physics_engine.operators import number_op
import numpy as np
c = cfg  # reuse previous cfg for N and hbar/omega
H = build_H(N=c.N, hbar=c.hbar, omega=c.omega, phi=c.phi, F=c.F, g=c.g, h=c.h)
evals, evecs = np.linalg.eigh(H)
# analytic energies for harmonic oscillator (approx): E_n = hbar*omega*(n+1/2)
n_vals = np.arange(min(len(evals), 10))
analytic = c.hbar * c.omega * (n_vals + 0.5)
print('numeric lowest eigenvalues (first 6):', np.round(evals[:6], 6))
print('analytic HO energies (first 6):', np.round(analytic[:6], 6))
rel_err = np.abs(evals[:6] - analytic[:6]) / (np.abs(analytic[:6]) + 1e-12)
print('relative error (first 6):', np.round(rel_err, 6))

In [ ]:
# Symbolic Noether-style check using SymPy (commutator with number operator)
try:
    from sympy import symbols, Function, simplify
    import sympy as sp
    print('SymPy available:', sp.__version__)
    # Use the engine's small helper to run a commutator magnitude check
    cfg_s = EngineConfig(N=12)
    eng_s = PhysicsEngine(cfg_s)
    print('Symbolic check (approx):', eng_s.symbolic_check())
except Exception as e:
    print('SymPy not available or check failed:', e)

In [ ]:
# Quick plots if matplotlib is present
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,3))
    plt.subplot(1,2,1)
    plt.plot(times, res.energies, '-o')
    plt.title('Energy (NumPy path)')
    plt.subplot(1,2,2)
    plt.plot(times, res.norms, '-o')
    plt.title('Norm (NumPy path)')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print('Plotting skipped (matplotlib likely missing):', e)

## Benchmark Results

The harmonic-oscillator benchmark was run; see [BENCHMARK_RESULTS.md](BENCHMARK_RESULTS.md) or [benchmark_results.ipynb](benchmark_results.ipynb) for numeric outputs and run instructions.